In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

%run MLProject.ipynb

df = pd.read_csv('usgs_main.csv')
df1 = df.copy()
df1['time'] = pd.to_datetime(df1['time'])
df1 = df1.sort_values('time')
df1 = df1.dropna(subset=['latitude','longitude','depth','mag','time'])


print(f"Başlangıç: {df1['time'].min()}")
print(f"Bitiş: {df1['time'].max()}")
print(f"Süre: {df1['time'].max() - df1['time'].min()}")
print()

dfweek_mlp = df1.set_index('time').resample('D').apply({
    'mag':'mean',
    'latitude':'mean',
    'longitude':'mean',
    'depth':'mean',
    'magType': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0],
    'type': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0],
    'status': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0],
    'net': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0]
})

dfweek_mlp = dfweek_mlp.reset_index(drop=True)
dfweek_mlp.index = dfweek_mlp.index + 1
dfweek_mlp.index.name = 'timeindex'


Başlangıç: 2022-03-03 21:37:08.970000+00:00
Bitiş: 2022-12-12 22:36:07.230000+00:00
Süre: 284 days 00:58:58.260000



KLASİK REGRESYONLA TAHMİN

In [ ]:
# Outlier detection
numerical_cols = ['mag', 'latitude', 'longitude', 'depth']
outlier_detector = IsolationForest(
    contamination=0.1,
    random_state=42,
    n_estimators=100
)

outlier_mask = outlier_detector.fit_predict(dfweek_mlp[numerical_cols]) == 1
dfweek_mlp = dfweek_mlp[outlier_mask]
print(f"Outlier detection sonrası veri sayısı: {len(dfweek_mlp)}")
print(f"Temizlenen outlier sayısı: {np.sum(~outlier_mask)}")

# Robust scaling
robust_scaler = RobustScaler()
robust_cols = ['mag', 'latitude', 'longitude', 'depth']
available_robust_cols = [col for col in robust_cols if col in dfweek_mlp.columns]

if available_robust_cols:
    robust_scaled_data = robust_scaler.fit_transform(dfweek_mlp[available_robust_cols])
    for i, col in enumerate(available_robust_cols):
        dfweek_mlp[f'{col}_robust'] = robust_scaled_data[:, i]


dfweek_mlp['futuremag'] = dfweek_mlp['mag'].shift(-1)
dfweek_mlp['futuredepth'] = dfweek_mlp['depth'].shift(-1)
dfweek_mlp['futurelat'] = dfweek_mlp['latitude'].shift(-1)
dfweek_mlp['futurelon'] = dfweek_mlp['longitude'].shift(-1)
dfweek_mlp = dfweek_mlp.dropna(subset=['futuremag','futuredepth','futurelat','futurelon'])


mlp_hiperparametreler = {
    'hidden_layer_sizes': [(50,), (100,),(150,), (50, 50), (100, 50),(150,100), (100, 100)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'lbfgs'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01, 0.1],
    'max_iter': [500, 1000, 1500],
    'shiftnum': [2, 3, 4]
}

train_mlp = dfweek_mlp[:int((4*len(dfweek_mlp))/5)]
test_mlp = dfweek_mlp[int(4*len(dfweek_mlp)/5):]


splitter_mlp = TimeSeriesSplit(n_splits=3)
grid_mlp = GridSearchCV(
    estimator=MLPEarthquakePredictor(random_state=42),
    param_grid=mlp_hiperparametreler,
    cv=splitter_mlp,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
grid_mlp.fit(train_mlp)
best_model_mlp = grid_mlp.best_estimator_

print(f"En iyi parametreler: {grid_mlp.best_params_}")

test_predictions_mlp = best_model_mlp.predict(test_mlp)
test_actual_mlp = test_mlp[['futuremag', 'futuredepth', 'futurelat', 'futurelon']].dropna()
min_len_mlp = min(len(test_predictions_mlp), len(test_actual_mlp))

target_names = ['futuremag', 'futuredepth', 'futurelat', 'futurelon']
target_labels = ['Magnitude', 'Depth', 'Latitude', 'Longitude']

for i, (name, label) in enumerate(zip(target_names, target_labels)):
    if i < test_predictions_mlp.shape[1] and i < test_actual_mlp.shape[1]:
        target_mse = mean_squared_error(
            test_actual_mlp.iloc[:min_len_mlp, i], 
            test_predictions_mlp[:min_len_mlp, i]
        )
        target_mae = mean_absolute_error(
            test_actual_mlp.iloc[:min_len_mlp, i], 
            test_predictions_mlp[:min_len_mlp, i]
        )
        target_r2 = r2_score(
            test_actual_mlp.iloc[:min_len_mlp, i], 
            test_predictions_mlp[:min_len_mlp, i]
        )
        
        print(f"   {label:12}: MSE={target_mse:.3f}, MAE={target_mae:.3f}, R²={target_r2:.3f}")

Outlier detection sonrası veri sayısı: 256
Temizlenen outlier sayısı: 29
Fitting 3 folds for each of 2268 candidates, totalling 6804 fits
En iyi parametreler: {'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (50,), 'learning_rate_init': 0.001, 'max_iter': 500, 'shiftnum': 2, 'solver': 'adam'}

 KLASİK REGRESYON 
   Magnitude   : MSE=0.034, MAE=0.145, R²=-0.607
   Depth       : MSE=12.404, MAE=2.756, R²=-0.109
   Latitude    : MSE=2.251, MAE=1.166, R²=0.052
   Longitude   : MSE=28.438, MAE=4.546, R²=-0.216


EŞİĞE GÖRE BAŞARIM ÖLÇÜMÜ

In [10]:
#Eşik değere göre tahmin ekledim
magnitude_thresholds = [1.5,2.0, 2.5, 3.0, 3.5, 4.0]

for threshold in magnitude_thresholds:
    print(f"\n Büyüklük Eşiği: {threshold}")
    
    # Tahmin edilen ve gerçek büyüklükler
    predicted_magnitudes = test_predictions_mlp[:min_len_mlp, 0] 
    actual_magnitudes = test_actual_mlp.iloc[:min_len_mlp, 0].values
    #Threshold a göre 
    predicted_earthquake = (predicted_magnitudes >= threshold).astype(int)
    actual_earthquake = (actual_magnitudes >= threshold).astype(int)
    accuracy = (predicted_earthquake == actual_earthquake).mean()
    # Detaylı istatistikler
    total_samples = len(actual_earthquake)
    actual_earthquakes = actual_earthquake.sum()
    predicted_earthquakes = predicted_earthquake.sum()
    correct_predictions = (predicted_earthquake == actual_earthquake).sum()
    
    print(f"Toplam örnek sayısı: {total_samples}")
    print(f"Gerçek deprem sayısı (>={threshold}): {actual_earthquakes}")
    print(f"Tahmin edilen deprem sayısı (>={threshold}): {predicted_earthquakes}")
    print(f"Doğru tahmin sayısı: {correct_predictions}")
    print(f"Doğruluk (Accuracy): {accuracy:.3f}")
    #confusion matrisde hata aldığım bir deneme olmuştu ve ben de düzeltip ne olur ne olmaz diyerek yapay zeka yardımıyla try-except ekledim  
    try:
        cm = confusion_matrix(actual_earthquake, predicted_earthquake)
        if cm.size == 4:  # 2x2 matrix
            tn, fp, fn, tp = cm.ravel()
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"Kesinlik (Precision): {precision:.3f}")
            print(f"Duyarlılık (Recall): {recall:.3f}")
            print(f"F1-Score: {f1_score:.3f}")
            print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
        else:
            print("Tek sınıf mevcut, detaylı metrikler hesaplanamıyor")
    except:
        print("Confusion matrix hesaplanamıyor")


 Büyüklük Eşiği: 1.5
Toplam örnek sayısı: 49
Gerçek deprem sayısı (>=1.5): 44
Tahmin edilen deprem sayısı (>=1.5): 47
Doğru tahmin sayısı: 42
Doğruluk (Accuracy): 0.857
Kesinlik (Precision): 0.894
Duyarlılık (Recall): 0.955
F1-Score: 0.923
Confusion Matrix: TN=0, FP=5, FN=2, TP=42

 Büyüklük Eşiği: 2.0
Toplam örnek sayısı: 49
Gerçek deprem sayısı (>=2.0): 0
Tahmin edilen deprem sayısı (>=2.0): 0
Doğru tahmin sayısı: 49
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

 Büyüklük Eşiği: 2.5
Toplam örnek sayısı: 49
Gerçek deprem sayısı (>=2.5): 0
Tahmin edilen deprem sayısı (>=2.5): 0
Doğru tahmin sayısı: 49
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

 Büyüklük Eşiği: 3.0
Toplam örnek sayısı: 49
Gerçek deprem sayısı (>=3.0): 0
Tahmin edilen deprem sayısı (>=3.0): 0
Doğru tahmin sayısı: 49
Doğruluk (Accuracy): 1.000
Tek sınıf mevcut, detaylı metrikler hesaplanamıyor

 Büyüklük Eşiği: 3.5
Toplam örnek sayısı: 49
Gerçek deprem 

KONUMA GÖRE BAŞARIM ÖLÇÜMÜ

In [ ]:
# Konuma göre
# Test verisini konum gruplarına böleriz,sırasıyla mag lat ve lon değerlerimiz
test_with_predictions = test_mlp.iloc[:min_len_mlp].copy()
test_with_predictions['predicted_mag'] = test_predictions_mlp[:min_len_mlp, 0]
test_with_predictions['predicted_lat'] = test_predictions_mlp[:min_len_mlp, 2]
test_with_predictions['predicted_lon'] = test_predictions_mlp[:min_len_mlp, 3]

# Daha büyük bölge grupları oluştur (1.0 derece aralıklarla)
test_with_predictions['lat_group'] = np.round(test_with_predictions['latitude'])
test_with_predictions['lon_group'] = np.round(test_with_predictions['longitude'])#alttaki satırda yapay zeka yardım etti
test_with_predictions['location_group'] = test_with_predictions['lat_group'].astype(str) + '_' + test_with_predictions['lon_group'].astype(str)

# Her konum grubu için değerlendirme
location_groups = test_with_predictions.groupby('location_group').size()
valid_locations = location_groups[location_groups >= 2].index  # En az 2 veri noktası

print(f"Yeterli veri olan bölge sayısı: {len(valid_locations)}")

threshold = 3.0  # Sabit eşik ile konum bazlı değerlendirme

for location in valid_locations[:10]:  #En az 2 veri olan 10 noktaya baktık
    location_data = test_with_predictions[test_with_predictions['location_group'] == location]
    lat, lon = location.split('_')#yine yapay zeka eklemesi
    
    # Bölgede threshold'dan büyük deprem var mı bakıyoruz
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    
    # Doğru tahmin mi?
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    #Burada görsel olarak güzel gözükmesi için yapay zekaya yazdırdım.
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()


import joblib
import json

joblib.dump(best_model_mlp, 'models/bestresult.pkl')
best_results = {
    'best_result': grid_mlp.best_params_,
    'evaluation_info': {
        'time_prediction_window': '1_week',
        'threshold_evaluation': 'added',
        'location_based_evaluation': 'added'
    }
}
with open('models/best_model_mlp.json', 'w') as f:
    json.dump(best_results, f, indent=2)



Yeterli veri olan bölge sayısı: 8
Bölge (36.0°, -120.0°) - Veri sayısı: 2
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Gerçek büyüklük: 1.82
  Tahmin büyüklük: 1.67

Bölge (37.0°, -110.0°) - Veri sayısı: 2
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Gerçek büyüklük: 1.91
  Tahmin büyüklük: 1.62

Bölge (37.0°, -117.0°) - Veri sayısı: 2
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Gerçek büyüklük: 1.67
  Tahmin büyüklük: 1.61

Bölge (38.0°, -111.0°) - Veri sayısı: 2
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Gerçek büyüklük: 1.91
  Tahmin büyüklük: 1.63

Bölge (38.0°, -115.0°) - Veri sayısı: 2
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Gerçek büyüklük: 1.73
  Tahmin büyüklük: 1.69

Bölge (38.0°, -116.0°) - Veri sayısı: 3
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Gerçek büyüklük: 1.70
  Tahmin büyüklük: 1.62

Bölge (39.0°, -112.0°) - Veri sayısı: 2
  Gerçek: Deprem yok
  Tahmin: D

EK OLARAK HER TAHMİNİN DOĞRU OLMADIĞI BİR THRESHOLD İÇİN BAKALIM

In [11]:
threshold = 1.70  # Sabit eşik ile konum bazlı değerlendirme

for location in valid_locations[:10]:  #En az 2 veri olan 10 noktaya baktık
    location_data = test_with_predictions[test_with_predictions['location_group'] == location]
    lat, lon = location.split('_')#yine yapay zeka eklemesi
    
    # Bölgede threshold'dan büyük deprem var mı bakıyoruz
    has_actual_earthquake = (location_data['futuremag'] >= threshold).any()
    has_predicted_earthquake = (location_data['predicted_mag'] >= threshold).any()
    
    # Doğru tahmin mi?
    correct_prediction = has_actual_earthquake == has_predicted_earthquake
    #Burada görsel olarak güzel gözükmesi için yapay zekaya yazdırdım.
    print(f"Bölge ({lat}°, {lon}°) - Veri sayısı: {len(location_data)}")
    print(f"  Gerçek: {'Deprem var' if has_actual_earthquake else 'Deprem yok'}")
    print(f"  Tahmin: {'Deprem var' if has_predicted_earthquake else 'Deprem yok'}")
    print(f"  Doğru tahmin: {'✓' if correct_prediction else '✗'}")
    print(f"  Gerçek büyüklük: {location_data['futuremag'].max():.2f}")
    print(f"  Tahmin büyüklük: {location_data['predicted_mag'].max():.2f}")
    print()

Bölge (36.0°, -120.0°) - Veri sayısı: 2
  Gerçek: Deprem var
  Tahmin: Deprem yok
  Doğru tahmin: ✗
  Gerçek büyüklük: 1.82
  Tahmin büyüklük: 1.67

Bölge (37.0°, -110.0°) - Veri sayısı: 2
  Gerçek: Deprem var
  Tahmin: Deprem yok
  Doğru tahmin: ✗
  Gerçek büyüklük: 1.91
  Tahmin büyüklük: 1.62

Bölge (37.0°, -117.0°) - Veri sayısı: 2
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Gerçek büyüklük: 1.67
  Tahmin büyüklük: 1.61

Bölge (38.0°, -111.0°) - Veri sayısı: 2
  Gerçek: Deprem var
  Tahmin: Deprem yok
  Doğru tahmin: ✗
  Gerçek büyüklük: 1.91
  Tahmin büyüklük: 1.63

Bölge (38.0°, -115.0°) - Veri sayısı: 2
  Gerçek: Deprem var
  Tahmin: Deprem yok
  Doğru tahmin: ✗
  Gerçek büyüklük: 1.73
  Tahmin büyüklük: 1.69

Bölge (38.0°, -116.0°) - Veri sayısı: 3
  Gerçek: Deprem var
  Tahmin: Deprem yok
  Doğru tahmin: ✗
  Gerçek büyüklük: 1.70
  Tahmin büyüklük: 1.62

Bölge (39.0°, -112.0°) - Veri sayısı: 2
  Gerçek: Deprem yok
  Tahmin: Deprem yok
  Doğru tahmin: ✓
  Gerç